# Project 1 — Video Inference (Task 4)
This notebook covers **Task 4 (Video Inference)**:
- Create a short clip if no MP4 is available (build from dataset images)
- Run YOLO inference on video and save annotated output
- Report average FPS + qualitative observations


In [1]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [2]:
import ultralytics, torch, platform
print("Ultralytics:", ultralytics.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Python:", platform.python_version())


Ultralytics: 8.3.246
Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Python: 3.10.19


In [3]:
from ultralytics import YOLO

BEST_MODEL = r"runs\weld_yolov8s_640\weights\best.pt"
model = YOLO(BEST_MODEL)


## 1) Create a short welding clip from test images (if you don't have an mp4)
This creates `videos/weld_clip.mp4`.

In [5]:
from pathlib import Path
import cv2

DATA_ROOT = Path(r"C:\Users\golla\Downloads\Welding\The Welding Defect Dataset\The Welding Defect Dataset")
IMG_DIR = DATA_ROOT/"test"/"images"

# Output video folder
OUT_VIDEO_DIR = Path("videos")
OUT_VIDEO_DIR.mkdir(exist_ok=True)
VIDEO_PATH = OUT_VIDEO_DIR/"weld_clip.mp4"

# Collect images
imgs = []
for ext in ("*.jpg","*.jpeg","*.png"):
    imgs += sorted(list(IMG_DIR.glob(ext)))

print("Total test images found:", len(imgs))
assert len(imgs) > 0, "No test images found!"

# Use first 100 frames (or less if fewer exist)
N_FRAMES = min(100, len(imgs))
FPS = 10  # 10 frames/sec => 100 frames => 10 sec
imgs = imgs[:N_FRAMES]

# Read first image to set video size
first = cv2.imread(str(imgs[0]))
h, w = first.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(VIDEO_PATH), fourcc, FPS, (w, h))

for p in imgs:
    frame = cv2.imread(str(p))
    if frame is None:
        continue
    writer.write(frame)

writer.release()
print(" Created video:", VIDEO_PATH.resolve())
print(f"Frames: {N_FRAMES}, FPS: {FPS}, Duration ~ {N_FRAMES/FPS:.1f} sec")


Total test images found: 74
✅ Created video: C:\Users\golla\Welding Project\videos\weld_clip.mp4
Frames: 74, FPS: 10, Duration ~ 7.4 sec


## 2) Run inference on the video and save annotated output

In [6]:
from ultralytics import YOLO

BEST_MODEL = r"runs\weld_yolov8s_640\weights\best.pt"
model = YOLO(BEST_MODEL)

model.predict(
    source=str(VIDEO_PATH),
    save=True,
    conf=0.25,
    project="video_out",
    name="weld_yolov8s_preds"
)

print(" Saved annotated video to: video_out/weld_yolov8s_preds/")



WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/74) C:\Users\golla\Welding Project\videos\weld_clip.mp4: 640x640 (no detections), 66.5ms
video 1/1 (frame 2/74) C:\Users\golla\Welding Project\videos\weld_clip.mp4: 640x640 2 Defects, 72.3ms
video 1/1 (frame 3/74) C:\Users\golla\Welding Project\videos\weld_clip.mp4: 640x640 (no detections), 55.1ms
video 1/1 (frame 4/74) C:\Users\golla\Welding Project\videos\weld_clip.mp4: 640x640 1 Good Weld, 52.9ms
video 1/1 (frame 5/74) C:\Users\golla\W

## 3) Compute average FPS (use stream=True to avoid RAM accumulation)

In [8]:
import time, cv2

cap = cv2.VideoCapture(str(VIDEO_PATH))
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

t0 = time.time()
for _ in model.predict(source=str(VIDEO_PATH), conf=0.25, verbose=False, stream=True):
    pass
t1 = time.time()

secs = t1 - t0
fps = frames / secs if secs > 0 else 0

print(f"Frames: {frames}")
print(f"Total inference time: {secs:.2f} sec")
print(f"Average FPS: {fps:.2f}")


Frames: 74
Total inference time: 4.19 sec
Average FPS: 17.68


### Video Inference

The trained YOLOv8s detector was applied to a short welding clip (generated from the dataset test images).

The output video was saved with bounding boxes and class labels.

### Inference speed

On NVIDIA RTX 4050 Laptop GPU at 640×640 resolution, the detector achieved 16.65 FPS on a 74-frame clip.

### Qualitative observations

Bad Weld and Good Weld are detected more reliably than Defect.

Defect is challenging due to small size and low contrast, causing occasional missed detections and false positives on reflections/texture.